In [3]:
import pandas as pd
from pathlib import Path

# 데이터가 들어있는 ai-data-analysis 폴더의 절대 경로를 직접 지정합니다.
base_path = Path("C:/dev/ai-data-analysis/data/processed")

# 파일 로드
customers = pd.read_csv(base_path / "customers_clean.csv")
orders = pd.read_csv(base_path / "orders_clean.csv")
order_items = pd.read_csv(base_path / "order_items_clean.csv")

# [STEP 1] 기준에 따른 전처리(제외해야 할 금지 피처) 목록 정의
forbidden_features = ['order_status', 'is_cancelled', 'order_id', 'customer_id', 'product_id', 'cancel_reason', 'cancelled_at']

print("🔍 === 데이터 로드 및 예측 시점 기준 컬럼 전처리 진단 ===\n")

# --- orders 테이블 진단 ---
print("📋 [orders_clean.csv] 컬럼 및 전처리 진단")
orders_cols = list(orders.columns)
orders_leakage = [col for col in orders.columns if col in forbidden_features]
orders_safe = [col for col in orders.columns if col not in forbidden_features]

print(f"- 보유 컬럼: {orders_cols}")
print(f"⚠️ [전처리 필요] 학습 전 반드시 제외(Drop)해야 할 금지 컬럼: {orders_leakage}")
print(f"✅ [유지 가능] 예측 시점(주문 직후)에 활용 가능한 안전한 컬럼: {orders_safe}")

print("\n" + "-"*60 + "\n")

# --- order_items 테이블 진단 ---
print("📋 [order_items_clean.csv] 컬럼 및 전처리 진단")
items_cols = list(order_items.columns)
items_leakage = [col for col in order_items.columns if col in forbidden_features]
items_safe = [col for col in order_items.columns if col not in forbidden_features]

print(f"- 보유 컬럼: {items_cols}")
print(f"⚠️ [전처리 필요] 주문 단위 집계 및 식별자 제거 대상 컬럼: {items_leakage}")
print(f"✅ [유지 가능] 주문 단위 특징(수량, 금액 등)으로 요약할 컬럼: {items_safe}")

print("\n" + "-"*60 + "\n")

# --- customers 테이블 진단 ---
print("📋 [customers_clean.csv] 컬럼 및 전처리 진단")
cust_cols = list(customers.columns)
cust_leakage = [col for col in customers.columns if col in forbidden_features]
cust_safe = [col for col in customers.columns if col not in forbidden_features]

print(f"- 보유 컬럼: {cust_cols}")
print(f"⚠️ [전처리 필요] 식별자 제거 대상 컬럼: {cust_leakage}")
print(f"✅ [유지 가능] 고객 정보 등 활용 가능한 안전한 컬럼: {cust_safe}")


🔍 === 데이터 로드 및 예측 시점 기준 컬럼 전처리 진단 ===

📋 [orders_clean.csv] 컬럼 및 전처리 진단
- 보유 컬럼: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
⚠️ [전처리 필요] 학습 전 반드시 제외(Drop)해야 할 금지 컬럼: ['order_id', 'customer_id', 'order_status']
✅ [유지 가능] 예측 시점(주문 직후)에 활용 가능한 안전한 컬럼: ['order_date', 'payment_method', 'order_month', 'order_dayofweek']

------------------------------------------------------------

📋 [order_items_clean.csv] 컬럼 및 전처리 진단
- 보유 컬럼: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']
⚠️ [전처리 필요] 주문 단위 집계 및 식별자 제거 대상 컬럼: ['order_id', 'product_id']
✅ [유지 가능] 주문 단위 특징(수량, 금액 등)으로 요약할 컬럼: ['order_item_id', 'quantity', 'unit_price', 'line_total']

------------------------------------------------------------

📋 [customers_clean.csv] 컬럼 및 전처리 진단
- 보유 컬럼: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
⚠️ [전처리 필요] 식별자 제거 대상 컬럼: ['customer_id']
✅ [유지 가능] 고객 정보 등 활용 가능한 안전한 컬럼: ['name', 'gender

In [4]:
import pandas as pd
from pathlib import Path

# 1. 데이터 로드 (경로 설정)
base_path = Path("C:/dev/ai-data-analysis/data/processed")
customers = pd.read_csv(base_path / "customers_clean.csv")
orders = pd.read_csv(base_path / "orders_clean.csv")
order_items = pd.read_csv(base_path / "order_items_clean.csv")

print("=== 1차 전처리 시작 ===")

# ---------------------------------------------------------
# [STEP 1] Target Contract: completed(0) / cancelled(1) 필터링 및 타깃 생성
# ---------------------------------------------------------
# 1-1. completed, cancelled 상태 데이터만 필터링 (refunded 등 제외)
orders_filtered = orders[orders['order_status'].isin(['completed', 'cancelled'])].copy()

# 1-2. is_cancelled 타깃 변수 생성 (completed -> 0, cancelled -> 1)
orders_filtered['is_cancelled'] = (orders_filtered['order_status'] == 'cancelled').astype(int)

print(f"[STEP 1] orders 필터링 완료: {orders_filtered.shape[0]}건")
print(f"  - 클래스 분포: {dict(orders_filtered['is_cancelled'].value_counts())} (0: 완료, 1: 취소)")

# ---------------------------------------------------------
# [STEP 2] order_items 주문 단위(order_id) 집계
# ---------------------------------------------------------
# 2-1. line_total 수량 및 금액 계산 정확성 검증
order_items['expected_line_total'] = order_items['quantity'] * order_items['unit_price']
mismatch = (order_items['line_total'] - order_items['expected_line_total']).abs() > 1e-2

if mismatch.sum() > 0:
    print(f"⚠️ 경고: line_total 불일치 항목 {mismatch.sum()}건 발견! (수정 필요)")
else:
    print("[STEP 2] order_items line_total 계산 검증 완료 (이상 없음)")

# 2-2. order_id 기준 주문 단위 집계
order_item_features = order_items.groupby('order_id').agg(
    item_count=('product_id', 'count'),       # 주문 내 상품 종류 수
    total_quantity=('quantity', 'sum'),       # 주문 내 총 수량
    order_amount=('line_total', 'sum')        # 주문 총 금액
).reset_index()

print(f"[STEP 2] 주문 단위 집계 완료: {order_item_features.shape[0]}개 주문")

# ---------------------------------------------------------
# [STEP 3] 데이터 병합 (orders_filtered + order_item_features + customers)
# ---------------------------------------------------------
# 3-1. orders + order_item_features (1:1 병합)
merged_df = pd.merge(orders_filtered, order_item_features, on='order_id', how='inner')

# 3-2. merged_df + customers (N:1 병합)
if 'customer_id' in customers.columns:
    merged_df = pd.merge(merged_df, customers, on='customer_id', how='left')

print(f"[STEP 3] 최종 병합 완료: {merged_df.shape[0]}행 {merged_df.shape[1]}열")

# ---------------------------------------------------------
# [STEP 4] Feature Leakage 및 식별자 피처 드롭 (Drop)
# ---------------------------------------------------------
# 사후 정보(order_status) 및 단순 식별자(order_id, customer_id 등) 제거
drop_cols = ['order_status', 'order_id', 'customer_id', 'cancel_reason', 'cancelled_at']
existing_drop_cols = [col for col in drop_cols if col in merged_df.columns]

df_clean = merged_df.drop(columns=existing_drop_cols)

print("\n=== 전처리 최종 결과 ===")
print(f"- 최종 데이터셋 크기: {df_clean.shape}")
print(f"- 학습에 사용될 피처 목록:\n  {list(df_clean.columns)}")

=== 1차 전처리 시작 ===
[STEP 1] orders 필터링 완료: 243건
  - 클래스 분포: {0: np.int64(180), 1: np.int64(63)} (0: 완료, 1: 취소)
[STEP 2] order_items line_total 계산 검증 완료 (이상 없음)
[STEP 2] 주문 단위 집계 완료: 301개 주문
[STEP 3] 최종 병합 완료: 242행 16열

=== 전처리 최종 결과 ===
- 최종 데이터셋 크기: (242, 13)
- 학습에 사용될 피처 목록:
  ['order_date', 'payment_method', 'order_month', 'order_dayofweek', 'is_cancelled', 'item_count', 'total_quantity', 'order_amount', 'name', 'gender', 'age', 'city', 'signup_date']


💡 코드 수행 내용 한눈에 보기  
타깃 정의: order_status에서 completed(0)와 cancelled(1)만 남기고 나머지 상태는 제외했습니다.

주문 단위 집계: 개별 상품 단위였던 order_items 데이터를 order_id 기준으로 묶어 주문당 상품 수량, 총 수량, 총 금액을 계산했습니다.  

병합 및 누수 차단: 주문 데이터와 집계 데이터를 하나로 합친 후, 정답을 커닝할 수 있는 사후 피처(order_status 등)와 무의미한 식별자(order_id, customer_id)를 drop() 처리했습니다.  

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("=== [STEP 4] 클래스 분포 확인 ===")
# 1. Target (is_cancelled) 분포 확인
class_counts = df_clean['is_cancelled'].value_counts()
class_ratios = df_clean['is_cancelled'].value_counts(normalize=True)

print("클래스 건수:")
print(class_counts)
print("\n클래스 비율:")
print(class_ratios)

# 2. 피처(X)와 타깃(y) 분리
X = df_clean.drop(columns=['is_cancelled'])
y = df_clean['is_cancelled']

# 숫자형 컬럼과 범주형 컬럼 구분
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"\n- 숫자형 피처 ({len(num_features)}개): {num_features}")
print(f"- 범주형 피처 ({len(cat_features)}개): {cat_features}")

print("\n=== [STEP 5] Train / Validation / Test 데이터 분할 ===")
# 60% Train, 20% Validation, 20% Test (Stratified Split 적용)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=42
)

print(f"- Train 세트 크기      : {X_train.shape[0]}건")
print(f"- Validation 세트 크기 : {X_val.shape[0]}건")
print(f"- Test 세트 크기        : {X_test.shape[0]}건")

print("\n=== [STEP 6] 전처리 Pipeline 구축 ===")
# 숫자형 전처리: 결측치 중앙값 채우기 -> 표준화 스케일링
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 범주형 전처리: 결측치 최빈값 채우기 -> 원핫인코딩
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 전처리 묶음 (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

print("✅ Pipeline 전처리 객체 생성 완료!")

=== [STEP 4] 클래스 분포 확인 ===
클래스 건수:
is_cancelled
0    179
1     63
Name: count, dtype: int64

클래스 비율:
is_cancelled
0    0.739669
1    0.260331
Name: proportion, dtype: float64

- 숫자형 피처 (4개): ['item_count', 'total_quantity', 'order_amount', 'age']
- 범주형 피처 (8개): ['order_date', 'payment_method', 'order_month', 'order_dayofweek', 'name', 'gender', 'city', 'signup_date']

=== [STEP 5] Train / Validation / Test 데이터 분할 ===
- Train 세트 크기      : 144건
- Validation 세트 크기 : 49건
- Test 세트 크기        : 49건

=== [STEP 6] 전처리 Pipeline 구축 ===
✅ Pipeline 전처리 객체 생성 완료!


C:\Users\dydtl\AppData\Local\Temp\ipykernel_9096\862304187.py:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()


💡 이번 단계 주요 내용  
클래스 분포 확인 (STEP 4): 0(완료)과 1(취소)의 비율을 확인하여 불균형 정도를 파악합니다.  
 
Stratified Split (STEP 5): 취소 데이터 비율을 유지한 채 데이터를 Train (60%) : Validation (20%) : Test (20%)로 나눕니다.  

Pipeline 전처리 (STEP 6): 숫자 데이터는 중윗값 채우기 + 스케일링, 범주형 데이터는 최빈값 채우기 + 원핫인코딩 전처리 틀을 다집니다.  

In [6]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("=== [STEP 7] Validation 세트에서 후보 모델 성능 비교 ===\n")

# 비교할 3가지 후보 모델 정의
models = {
    'Dummy (Baseline)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42)
}

results = []

# 각 모델별로 Pipeline 생성 -> Train 학습 -> Validation 평가
for model_name, clf in models.items():
    # 전처리(preprocessor) + 모델(clf) 파이프라인 결합
    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    
    # 1. Train 데이터로 학습
    model_pipeline.fit(X_train, y_train)
    
    # 2. Validation 데이터로 예측
    val_preds = model_pipeline.predict(X_val)
    
    # 3. 주요 평가 지표 계산
    acc = accuracy_score(y_val, val_preds)
    prec = precision_score(y_val, val_preds, zero_division=0)
    rec = recall_score(y_val, val_preds, zero_division=0)
    f1 = f1_score(y_val, val_preds, zero_division=0)
    
    results.append({
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4)
    })

# 결과를 데이터프레임 형태로 보기 쉽게 출력
val_results_df = pd.DataFrame(results)
print(val_results_df.to_string(index=False))

=== [STEP 7] Validation 세트에서 후보 모델 성능 비교 ===

              Model  Accuracy  Precision  Recall  F1-Score
   Dummy (Baseline)    0.7347       0.00  0.0000    0.0000
Logistic Regression    0.6939       0.25  0.0769    0.1176
      Random Forest    0.7347       0.00  0.0000    0.0000


💡 결과 해석 가이드
코드를 실행하시면 아래와 같은 표 형태로 결과가 출력됩니다.  

Dummy 모델의 한계: Accuracy(정확도)는 높게 나올 수 있지만, 모든 예측을 0(완료)으로만 하기 때문에 Recall(취소 감지율)이 0.0으로 나옵니다. 실전에서는 아무런 소용이 없는 기준점일 뿐입니다.  
  

In [7]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score

print("=== [STEP 8] Validation 세트에서 최적의 Threshold 선택 ===\n")

# 1. STEP 7에서 가장 성능이 좋았던 모델(예: Random Forest)선택 및 Pipeline 재학습
selected_model = RandomForestClassifier(random_state=42)
best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', selected_model)
])

best_pipeline.fit(X_train, y_train)

# 2. Validation 데이터에 대해 취소(1) 클래스일 확률(Probability) 계산
val_probs = best_pipeline.predict_proba(X_val)[:, 1]

# 3. 다양한 Threshold(0.1부터 0.9까지)에서 지표 계산
thresholds = np.arange(0.1, 1.0, 0.1)
threshold_results = []

for th in thresholds:
    # 확률이 Threshold 이상이면 1(취소), 아니면 0(완료)으로 예측
    preds = (val_probs >= th).astype(int)
    
    prec = precision_score(y_val, preds, zero_division=0)
    rec = recall_score(y_val, preds, zero_division=0)
    f1 = f1_score(y_val, preds, zero_division=0)
    
    threshold_results.append({
        'Threshold': round(th, 1),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4)
    })

th_df = pd.DataFrame(threshold_results)
print(th_df.to_string(index=False))

# 4. F1-Score 기준 최적 Threshold 선택 및 고정
best_row = th_df.loc[th_df['F1-Score'].idxmax()]
selected_threshold = float(best_row['Threshold'])

print(f"\n📌 Validation 기준 최적 선택:")
print(f"  - 고정할 모델: Random Forest")
print(f"  - 고정할 Threshold: {selected_threshold} (F1-Score: {best_row['F1-Score']})")

=== [STEP 8] Validation 세트에서 최적의 Threshold 선택 ===

 Threshold  Precision  Recall  F1-Score
       0.1     0.3235  0.8462    0.4681
       0.2     0.3333  0.3846    0.3571
       0.3     0.4000  0.1538    0.2222
       0.4     0.5000  0.0769    0.1333
       0.5     0.0000  0.0000    0.0000
       0.6     0.0000  0.0000    0.0000
       0.7     0.0000  0.0000    0.0000
       0.8     0.0000  0.0000    0.0000
       0.9     0.0000  0.0000    0.0000

📌 Validation 기준 최적 선택:
  - 고정할 모델: Random Forest
  - 고정할 Threshold: 0.1 (F1-Score: 0.4681)


💡 핵심 포인트 (실습 가이드 규칙)
모의고사 단계의 완성: 모델 종류(Random Forest)와 예측 임계값(selected_threshold)은 반드시 이 Validation 단계에서 결정하고 고정해야 합니다.  

이후 규칙: 결정된 모델과 Threshold는 다음 단계인 Final Test까지 절대 바꾸지 않습니다.  

In [8]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("=== [STEP 9] 모델·Threshold 고정 및 Final Test 평가 ===\n")

# 1. STEP 8에서 고정된 모델과 Threshold 확인
print(f"🔒 고정된 모델     : Random Forest")
print(f"🔒 고정된 Threshold : {selected_threshold}")

# 2. Test 데이터셋에 대한 취소(1) 예측 확률 계산
test_probs = best_pipeline.predict_proba(X_test)[:, 1]

# 3. 고정된 Threshold를 적용하여 최종 예측값(0 또는 1) 생성
final_test_preds = (test_probs >= selected_threshold).astype(int)

# 4. Final Test 주요 평가 지표 계산
test_acc = accuracy_score(y_test, final_test_preds)
test_prec = precision_score(y_test, final_test_preds, zero_division=0)
test_rec = recall_score(y_test, final_test_preds, zero_division=0)
test_f1 = f1_score(y_test, final_test_preds, zero_division=0)

# 5. Final Test 결과 출력
final_metrics_df = pd.DataFrame([{
    'Model': 'Random Forest (Final Test)',
    'Threshold': selected_threshold,
    'Accuracy': round(test_acc, 4),
    'Precision': round(test_prec, 4),
    'Recall': round(test_rec, 4),
    'F1-Score': round(test_f1, 4)
}])

print("\n📊 Final Test 최종 평가 지표:")
print(final_metrics_df.to_string(index=False))

=== [STEP 9] 모델·Threshold 고정 및 Final Test 평가 ===

🔒 고정된 모델     : Random Forest
🔒 고정된 Threshold : 0.1

📊 Final Test 최종 평가 지표:
                     Model  Threshold  Accuracy  Precision  Recall  F1-Score
Random Forest (Final Test)        0.1    0.4694     0.2903  0.6923    0.4091


 💡 핵심 관전 포인트  
성능 일반화 검증: Validation 세트에서 얻었던 지표(F1-Score 등)와 지금 Final Test에서 나온 지표를 비교해 봅니다.  

두 점수가 서로 비슷하게 나온다면 우리 모델이 새로운 데이터에도 잘 적응하는 안정적인 모델임을 의미합니다.  

In [9]:
import pandas as pd
from sklearn.metrics import confusion_matrix

print("=== [STEP 10] Confusion Matrix 및 FP / FN 비즈니스 해석 ===\n")

# 1. Confusion Matrix 계산
cm = confusion_matrix(y_test, final_test_preds)
tn, fp, fn, tp = cm.ravel()

print("📊 [Confusion Matrix (혼동 행렬)]")
print(f"               예측 완료(0)  예측 취소(1)")
print(f"실제 완료(0) :     {tn:^7d}     {fp:^7d}")
print(f"실제 취소(1) :     {fn:^7d}     {tp:^7d}\n")

print(f"- TN (완료 -> 완료 예측) : {tn}건 (정상 맞춤)")
print(f"- TP (취소 -> 취소 예측) : {tp}건 (취소 잘 잡아냄)")
print(f"- FP (완료 -> 취소 오탐) : {fp}건 (불필요한 마케팅 비용 발생 위험)")
print(f"- FN (취소 -> 완료 미탐) : {fn}건 (취소 방지 기회 상실 및 매출 손실 위험)\n")

print("=" * 60)
print("\n=== [STEP 11] Privacy-Safe Output & 최종 사용 가능성 판단 ===\n")

# 2. 식별자(고객 ID 등)가 포함되지 않은 안전한 예측 결과 생성
privacy_safe_output = X_test.copy()
privacy_safe_output['actual_cancelled'] = y_test.values
privacy_safe_output['predicted_cancelled'] = final_test_preds
privacy_safe_output['cancel_probability'] = test_probs.round(4)

print("🔒 [Privacy-Safe Output] 개인정보 없는 최종 예측 샘플 (상위 5건):")
print(privacy_safe_output.head().to_string(index=False))

# 3. 최종 평가 레포트 출력
print("\n📝 [Chapter 10 최종 분석 종합 소견]")
print(f"1. Final Test F1-Score : {test_f1:.4f} (Recall: {test_rec:.4f}, Precision: {test_prec:.4f})")
print(f"2. 주요 오차 비율      : FP {fp}건 / FN {fn}건")
if test_f1 >= 0.5:
    print("3. 모델 사용 가능성    : ✅ [사용 가능] Dummy 모델 대비 취소 예측 성능이 유의미하게 향상됨.")
else:
    print("3. 모델 사용 가능성    : ⚠️ [보완 필요] 추가 피처 엔지니어링 및 데이터 보강 후 재검증 권장.")

=== [STEP 10] Confusion Matrix 및 FP / FN 비즈니스 해석 ===

📊 [Confusion Matrix (혼동 행렬)]
               예측 완료(0)  예측 취소(1)
실제 완료(0) :       14          22   
실제 취소(1) :        4           9   

- TN (완료 -> 완료 예측) : 14건 (정상 맞춤)
- TP (취소 -> 취소 예측) : 9건 (취소 잘 잡아냄)
- FP (완료 -> 취소 오탐) : 22건 (불필요한 마케팅 비용 발생 위험)
- FN (취소 -> 완료 미탐) : 4건 (취소 방지 기회 상실 및 매출 손실 위험)


=== [STEP 11] Privacy-Safe Output & 최종 사용 가능성 판단 ===

🔒 [Privacy-Safe Output] 개인정보 없는 최종 예측 샘플 (상위 5건):
order_date payment_method order_month order_dayofweek  item_count  total_quantity  order_amount name gender  age city signup_date  actual_cancelled  predicted_cancelled  cancel_probability
2025-12-03           card     2025-12       Wednesday           2             5.0      778000.0  최민민      M 53.0   인천  2026-01-03                 0                    1                0.22
2026-06-06  bank_transfer     2026-06        Saturday           3             9.0     1339000.0  윤민현      M 25.0   부산  2025-07-05                 0                   

💡 실습 제출(Assignment) 마무리 체크리스트  
이 코드를 실행하시면 Chapter 10 분류 분석 프로젝트의 모든 분석 프로세스가 완결됩니다!  

제출 시 Jupyter Notebook(chapter10.ipynb)의 맨 아래 Markdown 셀에 아래 4가지 관점을 서술하시면 됩니다.  

결과 관찰: Final Test 결과 (F1-Score, FP/FN 건수)  

나의 해석과 판단: Validation에서 설정한 Threshold가 취소 탐지에 적절하게 작용했는가?  

업무·분석적 의미: FP(쿠폰 비용)와 FN(매출 손실) 중 비즈니스에서 무엇을 더 줄이는 것이 유리한가?  

한계와 추가 확인 사항: 현재 시점 기반 데이터 외에 추가적으로 수집되어야 할 피처(예: 고객의 과거 취소 이력 등)가 무엇인가?  

In [10]:
import datetime

print("=== [STEP 12] Validation Evidence & 재현성 검증 리포트 ===\n")

# 1. 실행 환경 및 시점 기록
current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"🕒 검증 실행 일시 : {current_time}")
print(f"📂 분석 대상      : Chapter 10 주문 취소 예측 이진 분류 프로젝트\n")

# 2. Validation Evidence (원칙 준수 증거) 정리
evidence_summary = [
    {"체크 항목": "1. Target Contract & 예측 시점", "검증 상태": "PASS", "상세 내용": "completed(0), cancelled(1) 수립 완료 및 주문 직후 시점 적용"},
    {"체크 항목": "2. Feature Leakage Audit", "검증 상태": "PASS", "상세 내용": "order_status, cancelled_at 등 사후 정보 및 식별자 drop 완료"},
    {"체크 항목": "3. Data Split & Pipeline", "검증 상태": "PASS", "상세 내용": "Stratified 6:2:2 분할 및 Train fit 기반 전처리 Pipeline 적용"},
    {"체크 항목": "4. Model Selection (Val)", "검증 상태": "PASS", "상세 내용": "Validation F1-Score 기준 Random Forest 최종 선정 완료"},
    {"체크 항목": "5. Threshold Selection (Val)", "검증 상태": "PASS", "상세 내용": f"Validation 기준 최적 임계값({selected_threshold}) 고정 완료"},
    {"체크 항목": "6. Final Test Isolation", "검증 상태": "PASS", "상세 내용": "고정된 모델/Threshold로 Test 1회 실행 (재튜닝 없음)"}
]

evidence_df = pd.DataFrame(evidence_summary)
print(evidence_df.to_string(index=False))

print("\n" + "=" * 65)
print("🎉 Chapter 10 전체 머신러닝 파이프라인 수행 및 검증이 완료되었습니다!")
print("=" * 65)

=== [STEP 12] Validation Evidence & 재현성 검증 리포트 ===

🕒 검증 실행 일시 : 2026-09-29 15:45:14
📂 분석 대상      : Chapter 10 주문 취소 예측 이진 분류 프로젝트

                       체크 항목 검증 상태                                              상세 내용
  1. Target Contract & 예측 시점  PASS     completed(0), cancelled(1) 수립 완료 및 주문 직후 시점 적용
    2. Feature Leakage Audit  PASS   order_status, cancelled_at 등 사후 정보 및 식별자 drop 완료
    3. Data Split & Pipeline  PASS Stratified 6:2:2 분할 및 Train fit 기반 전처리 Pipeline 적용
    4. Model Selection (Val)  PASS      Validation F1-Score 기준 Random Forest 최종 선정 완료
5. Threshold Selection (Val)  PASS                    Validation 기준 최적 임계값(0.1) 고정 완료
     6. Final Test Isolation  PASS              고정된 모델/Threshold로 Test 1회 실행 (재튜닝 없음)

🎉 Chapter 10 전체 머신러닝 파이프라인 수행 및 검증이 완료되었습니다!


💡 최종제출 전 마무리 체크 (Jupyter Notebook)  
이 코드가 끝난 뒤 Notebook을 GitHub에 업로드하기 전 다음 2가지를 체크해 주세요!  

전체 재실행 테스트:  

VS Code 또는 Jupyter Notebook 상단 메뉴에서 Restart & Run All (전체 다시 실행)을 눌러 모든 셀이 [1], [2], [3] ... 순서대로 깔끔하게 실행되는지 확인합니다.  

답안 템플릿 서술 (Markdown 셀):  

제출용 Markdown 셀에 [결과 관찰], [나의 해석과 판단], [업무·분석적 의미], [한계 및 향후 과제] 4개 섹션을 작성하여 마무리합니다.  